In [ ]:
# Cell 1 — Setup
"""
05_baselines.ipynb
==================
Interactive baseline notebook focused on DirectGNN and descriptor-model comparison.

CLI equivalents and related scripts:
- `python scripts/training/train_directgnn.py ...`
- `python scripts/experiments/run_seeds.py --train-script scripts/training/train_directgnn.py ...`
- `python scripts/external/run_fastsolv.py ...`
- `python scripts/experiments/run_split_comparisons.py --splits "solute_scaffold,solute,solvent" ...`
- `python scripts/experiments/learning_curves.py --models "tgnn_solv,direct_gnn,rf_baseline" ...`
- `python scripts/experiments/statistical_tests.py --results ... --labels ...`
- `src/tgnn_solv/baselines/rf_baseline.py` and `ideal_sle.py` for classical baselines
"""

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"
TABLES_DIR = PROJECT_ROOT / "tables"
NOTEBOOK_FIG_DIR = FIGURES_DIR / "notebooks"
NOTEBOOK_RESULTS_DIR = RESULTS_DIR / "notebooks"
NOTEBOOK_FIG_DIR.mkdir(parents=True, exist_ok=True)
NOTEBOOK_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import torch
import pandas as pd
import matplotlib.pyplot as plt

from tgnn_solv.config import TGNNSolvConfig
from tgnn_solv.inference import load_model
from tgnn_solv.data import make_loaders, PROCESSED_DIR
from tgnn_solv.baselines import run_baseline, compare_with_tgnn

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
print(f"Project root: {PROJECT_ROOT}")
print(f"Device: {DEVICE}")


## The mathematics behind the baseline comparisons

This notebook compares two different hypotheses about where the task complexity should live.
For DirectGNN, the prediction is made directly:

$$
\widehat y_{\mathrm{Direct}} = f_\theta\!\left(g_{\mathrm{sol}}, g_{\mathrm{slv}}, \phi(T)\right),
\qquad y = \ln x_2.
$$

If descriptor augmentation is enabled, the pair representation is expanded with a symmetric descriptor block:

$$
d_{\mathrm{pair}} =
\left[
d_{\mathrm{sol}},
d_{\mathrm{slv}},
d_{\mathrm{sol}} \odot d_{\mathrm{slv}},
\left|d_{\mathrm{sol}} - d_{\mathrm{slv}}\right|
\right].
$$

The augmented direct baseline is therefore

$$
\widehat y_{\mathrm{Direct+Desc}} =
f_\theta\!\left(g_{\mathrm{sol}}, g_{\mathrm{slv}}, \phi(T), d_{\mathrm{pair}}\right).
$$

For TGNN-Solv, in contrast, direct prediction is not the main path;
it only appears as a bounded correction on top of the physics bottleneck:

$$
\widehat y_{\mathrm{TGNN}} =
\ln x_2^{\mathrm{phys}} + (1-c)\,r.
$$

The RF baseline can be viewed as an ensemble of trees over hand-crafted features \($\psi(x)\$):

$$
\widehat y_{\mathrm{RF}}(x) = \frac{1}{B} \sum_{b=1}^{B} T_b\!\left(\psi(x)\right),
$$

where \($\psi(x)$\) is either the RDKit descriptor vector, Morgan fingerprints, or a hybrid of both.


## Step 1. Fix the split and the comparison conditions

Before running the baselines it is important to be explicit about the split.
Otherwise the difference between models gets mixed together with the difference
between generalization protocols.


In [ ]:
# Cell 2 — Load data
# Baseline work is usually reported on the random-by-solute split.
# Switch back to train.csv / val.csv / test.csv if you want the strict scaffold split.
train_df = pd.read_csv(PROCESSED_DIR / "train_solute.csv")
val_df = pd.read_csv(PROCESSED_DIR / "val_solute.csv")
test_df = pd.read_csv(PROCESSED_DIR / "test_solute.csv")

cfg = TGNNSolvConfig(
    hidden_dim=256,
    n_gnn_layers=6,
    encoder_role_mode="shared_residual",
    encoder_role_specific_layers=2,
    n_cross_attn_layers=3,
    n_attn_heads=8,
    pair_dim=512,
    nrtl_tau_mode="ref_invT",
    use_pair_temperature_batching=True,
    pair_temperature_min_group_size=2,
    pair_temperature_group_chunk_size=4,
)

train_loader, val_loader, test_loader = make_loaders(
    train_df,
    val_df,
    test_df,
    batch_size=cfg.batch_size,
    use_pair_temperature_batching=cfg.use_pair_temperature_batching,
    pair_temperature_min_group_size=cfg.pair_temperature_min_group_size,
    pair_temperature_group_chunk_size=cfg.pair_temperature_group_chunk_size,
)


## Step 2. Train the no-physics baseline

This block answers the main control question: how far can the model get if we
remove the solver and train the same backbone directly on `ln(x_2)`? That is why
DirectGNN is the main maintained baseline rather than a side experiment.


In [ ]:
# Cell 3 — Train DirectGNN baseline
baseline_metrics = run_baseline(
    train_loader, val_loader, test_loader,
    cfg=cfg, device=DEVICE,
    n_epochs=2, patience=20,
)

## Step 3. Compare the physics path against the direct path

After training the baseline, it is useful to inspect not only the absolute metric,
but also the error profile relative to TGNN-Solv: where the physics bottleneck truly helps,
and where direct prediction catches up or even wins.


In [ ]:
# Cell 4 — Compare with TGNN-Solv
MODEL_PATH = CHECKPOINT_DIR / "tgnn_solv_trained.pt"
model, model_cfg = load_model(str(MODEL_PATH), DEVICE)

comparison = compare_with_tgnn(
    model, model_cfg, baseline_metrics,
    test_loader, test_df,
)

comparison.to_csv(NOTEBOOK_RESULTS_DIR / "baseline_comparison.csv", index=False)
comparison


## Step 4. A visual readout of the differences

The plots below turn the tabular comparison into a more readable picture. This is often
where it becomes clear whether the benefit of physics is uniform or concentrated in specific
subsets of systems.


In [ ]:
# Cell 5 — Visualization

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

names = comparison["name"].values
colors = ["coral", "steelblue"]

for ax, metric, title, better in [
    (axes[0], "mae", "MAE (ln x₂) ↓", "lower"),
    (axes[1], "rmse", "RMSE (ln x₂) ↓", "lower"),
    (axes[2], "r2", "R² ↑", "higher"),
]:
    vals = comparison[metric].values
    bars = ax.bar(names, vals, color=colors, width=0.5, edgecolor="black")
    ax.set_title(title, fontsize=12)
    ax.set_ylabel(metric.upper())

    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                f"{val:.3f}", ha="center", va="bottom", fontsize=11)

plt.tight_layout()
plt.savefig(NOTEBOOK_FIG_DIR / "baseline_comparison.png", dpi=150)
plt.show()


## Step 5. Sanity check the temperature feature

In DirectGNN, temperature is injected explicitly without the solver structure. It is therefore
useful to check separately that the temperature encoding is not degenerate and that the model is
actually using temperature information rather than ignoring it.


In [ ]:
# Cell 6 — Temperature encoding sanity check

from tgnn_solv.baselines.temperature import ThermometerEncoder

enc = ThermometerEncoder(n_bins=10, T_min=200, T_max=500)

test_temps = torch.tensor([250.0, 298.15, 350.0, 450.0])
encoded = enc.encode(test_temps)

print("Temperature encoding examples (10 bins, 200-500 K):")
print(f"Bin width: {enc.bin_width:.0f} K")
for i, T_val in enumerate(test_temps):
    vals = [f"{v:.2f}" for v in encoded[i].tolist()]
    print(f"  T={T_val.item():6.1f} K → [{', '.join(vals)}]")